# 🏎️ Laboratorio di Reinforcement Learning: Missione Hill Climb Racing
### Progettazione della Funzione di Reward (Reward Engineering)

Benvenuti nel laboratorio di Reinforcement Learning su **Hill Climb Racing** (Box2D Physics Engine).  
In questa attività sperimentale progetterai la formulazione matematica della **funzione di Reward** per guidare un pilota artificiale (agente PPO) ad apprendere in modo autonomo come scalare una catena di colline impervie fino al traguardo degli **800 metri** senza ribaltarsi o spezzarsi il collo.

---


## 1. L'idea: imparare per tentativi ed errori

Nel **Reinforcement Learning** (apprendimento per rinforzo) l'Intelligenza Artificiale non riceve istruzioni su *cosa* fare passo dopo passo, ma impara da sola esplorando l'ambiente e cercando di massimizzare un punteggio cumulativo chiamato **Reward**. Ci sono due protagonisti:

| Protagonista | Chi è nel nostro laboratorio |
|---|---|
| **Agente** (chi impara) | Il "pilota" artificiale (algoritmo **PPO**) che decide quando accelerare, frenare o lasciare l'auto in folle |
| **Environment** (il mondo) | Il simulatore fisico *Hill Climb Racing*: gravità, pendenze collinari procedurali, sospensioni e pilota articolato |

Il simulatore fisico Box2D gira a 60 FPS con frame skip 3x: l'agente prende **20 decisioni al secondo** (ogni *step* dura esattamente $0{,}05\text{ secondi}$ simulati):

```text
        ┌────────────────  osservazione obs (8 sensori fisici di bordo)  ────────────────┐
        ▼                                                                                 │
   ┌──────────┐  azione action (0: folle, 1: acceleratore, 2: freno/retro)   ┌─────────────────────────┐
   │  AGENTE  │ ───────────────────────────────────────────────────────────▶ │       ENVIRONMENT       │
   │ (pilota) │                                                              │ (vettura + fisica Box2D)│
   └──────────┘ ◀─────────────  REWARD (premio + o penalità -)  ──────────── └─────────────────────────┘
```

**Attenzione alla regola d'oro del Reinforcement Learning:**  
L'agente non sa cosa sia una collina o cosa significhi guidare bene. Conosce **solo i numeri che gli fornisci tu**! Il suo unico obiettivo matematico è **massimizzare la somma totale delle reward ricevute dall'inizio alla fine dell'episodio**:
$$R_{\text{totale}} = r_1 + r_2 + r_3 + \dots + r_T$$
Se scrivi una formula sbilanciata, l'agente troverà scorciatoie assurde (*Reward Hacking*) per accumulare punti senza arrivare al traguardo (ad esempio impennandosi e schiantandosi subito per fare prima)!


### 🛠️ Installazione e preparazione del motore di simulazione su Google Colab
Esegui le due celle seguenti per installare le librerie necessarie e scaricare il pacchetto didattico dalla repository ufficiale:
👉 **https://github.com/EmanueleCiccarelli/RL_school**


In [ ]:
# 1. Installazione librerie di sistema e pacchetti Python su Colab
!apt-get update -qq
!apt-get install -y -qq swig ffmpeg
!pip install -q "gymnasium[box2d]" stable-baselines3 pygame imageio imageio-ffmpeg tqdm rich fpdf2 matplotlib

# 2. Download del pacchetto dalla repo GitHub ufficiale (se non già presente)
import os, sys
if not os.path.exists("play2learn_hillclimb"):
    !git clone -q https://github.com/EmanueleCiccarelli/RL_school.git
    if os.path.exists("RL_school"):
        %cd RL_school


In [ ]:
import play2learn_hillclimb
print(f"✅ Modulo play2learn_hillclimb v{getattr(play2learn_hillclimb, '__version__', '0.2.0')} caricato con successo!")


---
## 2. Conosci la Fisica della Scena e il Sistema di Riferimento

Per progettare una buona funzione matematica devi conoscere le misure esatte e le leggi fisiche dell'ambiente:

* **Origine e Partenza (Spawn)**: La vettura appare all'inizio del tracciato collinare a circa $x \approx 6{,}67\text{ metri}$.
* **Traguardo di Successo ($800\text{ metri}$)**: L'obiettivo della scalata è raggiungere o superare gli **800 metri** di distanza percorsa (`dist_x >= 800.0`).
* **Morfologia del Terreno**: Le colline sono generate proceduralmente tramite onde di rumore continuo (Perlin noise deterministico). Più avanzi, più le salite si fanno ripide e irregolari!
* **Struttura della Vettura**:
  1. **Telaio principale**: corpo rigido con baricentro ribassato.
  2. **Sospensioni indipendenti**: pistoni elastici ammortizzati collegano il telaio ai mozzi delle ruote.
  3. **Due ruote motrici ad alta aderenza**: ciascuna ruota traccia in tempo reale il contatto col suolo (`rear_wheel` e `front_wheel`).
  4. **Pilota articolato (Torso + Testa)**: il busto e la testa del pilota oscillano con l'inerzia della marcia.
* **Condizione di Schianto Mortale (`is_crashed = True`)**:
  Se la vettura si capovolge all'indietro o in avanti e la **testa del pilota tocca terra**, il pilota si spezza il collo e l'episodio termina immediatamente con una grave penalità!

Esegui la cella seguente per visualizzare la **Mappa Fisica Annotata** dell'ambiente:


In [ ]:
play2learn_hillclimb.mostra_mappa_ambiente()


---
## 3. Sensori (`obs`), Motori (`action`) ed Eventi di Fine Corsa

Ad ogni singolo step ($0{,}05\text{ s}$), la tua funzione `custom_reward(obs, action, is_success, is_crashed, is_timeout)` riceve:

### A. Il vettore `obs` con gli 8 sensori fisici di bordo
| Indice | Sensore | Grandezza Fisica | Significato e Valori Tipici |
| :---: | :--- | :--- | :--- |
| **`obs[0]`** | **`dist_x`** | Distanza percorsa | Metri percorsi lungo la collina (Spawn $\approx 6{,}67\text{ m}$, Traguardo $\ge 800\text{ m}$) |
| **`obs[1]`** | **`quota_y`** | Quota altimetrica | Quota verticale del veicolo sul terreno (in metri) |
| **`obs[2]`** | **`vx`** | Velocità orizzontale | $v_x > 0$ marcia in avanti, $v_x < 0$ retromarcia o scivolamento all'indietro |
| **`obs[3]`** | **`vy`** | Velocità verticale | $v_y < 0$ la vettura sta salendo, $v_y > 0$ sta scendendo/cadendo |
| **`obs[4]`** | **`angle`** | Angolo di beccheggio | In radianti ($-\pi \dots +\pi$): `0.0` orizzontale, $> 0$ muso in alto, $< 0$ muso in basso. Oltre $\pm 1{,}4\text{ rad}$ ($\approx 80^\circ$) l'auto è a rischio ribaltamento! |
| **`obs[5]`** | **`ang_vel`** | Velocità angolare | Velocità di rotazione del telaio su se stesso in radianti al secondo |
| **`obs[6]`** | **`rear_wheel`**| Contatto ruota posteriore | Vale `1.0` se la gomma posteriore tocca terra con aderenza, `0.0` se è staccata in aria |
| **`obs[7]`** | **`front_wheel`**| Contatto ruota anteriore | Vale `1.0` se la gomma anteriore tocca terra con aderenza, `0.0` se è sollevata (impennata o salto) |

### B. Il comando motore `action` scelto dall'agente
| Valore `action` | Azione del Pilota | Effetto Fisico sul Veicolo | Quando usarlo |
| :---: | :--- | :--- | :--- |
| **`0`** | **Folle (Motori Spenti)** | Nessuna coppia applicata: la vettura prosegue per inerzia naturale | Nelle discese o a terra dopo un dosso |
| **`1`** | **Acceleratore (Gas)** | Applica coppia motrice in avanti su entrambe le ruote | Per scalare le salite o prendere velocità |
| **`2`** | **Freno / Retromarcia** | Applica coppia frenante e spinta contraria | Per abbassare il muso in volo ed evitare ribaltamenti all'indietro |

> 💡 **Fisica del volo balistico:**
> Quando l'auto decolla da una duna, accelerare (`action == 1`) in aria fa ruotare il telaio all'indietro (effetto giroscopico delle ruote in rotazione), favorendo il ribaltamento a testa in giù! Frenare (`action == 2`) in aria invece abbassa il muso e prepara l'atterraggio sulle quattro ruote!

### C. I 3 Flag Booleani di Fine Episodio
* **`is_success`** (`True`/`False`): Scalata completata con successo! L'auto ha superato il traguardo degli **800 metri**.
* **`is_crashed`** (`True`/`False`): Schianto letale! L'auto si è ribaltata e la testa del pilota ha toccato terra.
* **`is_timeout`** (`True`/`False`): Tempo massimo scaduto oppure veicolo rimasto bloccato senza riuscire a superare una salita per oltre 200 step.


### 🔍 Esplorazione: guarda i numeri in tempo reale
Esegui la cella seguente per simulare 10 step con azioni casuali e osservare come variano i valori fisici del veicolo.


In [ ]:
import gymnasium as gym
import numpy as np
import hill_racing_env
from play2learn_hillclimb.environment import HillClimbRewardWrapper

raw_env = gym.make("hill_racing_env/HillRacing-v0")
env = HillClimbRewardWrapper(raw_env)
obs, _ = env.reset(seed=42)

nomi_azioni = {
    0: "0: folle        ",
    1: "1: acceleratore ",
    2: "2: freno / retro",
}

print("═══════════════════════════════════════════════════════════════════════════════════════")
for step in range(10):
    azione = env.action_space.sample()
    obs, r_orig, term, trunc, info = env.step(azione)
    dist_x, quota_y, vx, vy, angle, ang_vel, rear_g, front_g = obs
    print(
        f"step {step:2d} | {nomi_azioni[int(azione)]} | "
        f"x={dist_x:6.1f}m y={quota_y:5.1f}m | vx={vx:5.1f} vy={vy:5.1f} | "
        f"ang={angle:5.2f} rad ({np.degrees(angle):5.1f}°) | "
        f"ruote=[post:{int(rear_g)}, ant:{int(front_g)}]"
    )
print("═══════════════════════════════════════════════════════════════════════════════════════")

env.close()


---
## 4. La Matematica dei Premi e le 2 Trappole Classiche 🧠

Prima di scrivere la tua funzione, tieni a mente questi **dati di riferimento temporali**:
* 💥 **Ribaltamento e schianto immediato (Impennata)**: dura circa **30 – 60 step** ($1{,}5 - 3{,}0\text{ s}$).
* 🎯 **Scalata completa verso gli 800m**: dura circa **300 – 700 step** ($15 - 35\text{ s}$).
* ⏳ **Blocco in salita / Timeout**: scatta se la vettura resta piantata senza avanzare.

Poiché la tua funzione `custom_reward` viene chiamata **ad ogni singolo step**, una penalità o un premio ripetuto per centinaia di step determina il destino dell'apprendimento! Ecco le **2 trappole matematiche classiche** da evitare:

### ⚠️ Trappola 1: Il "Gas Fisso a Tavoletta" (Backflip suicida)
* **L'errore:** Dai un premio enorme solo per la distanza percorsa `dist_x` o per la velocità in avanti `vx`, senza penalizzare l'eccessiva inclinazione del muso.
* **Il calcolo dell'agente:** *"Più premo gas, più faccio punti velocemente!"* Il veicolo accelera alla massima potenza sul primo dosso, decolla all'indietro, compie una rotazione di 360° e si spacca la testa al suolo in 40 step.
* **Come risolverla:** Aggiungi penalità progressive se l'inclinazione `abs(angle)` o la rotazione `abs(ang_vel)` aumentano troppo, e disincentiva il gas continuo quando le ruote sono entrambe staccate da terra!

### ⚠️ Trappola 2: L'"Oscillatore Codardo" (Paura delle salite)
* **L'errore:** Metti una penalità di schianto gigantesca (`reward -= 500.0`) e forti penalità per qualsiasi oscillazione o inclinazione, senza premiare l'avanzamento costante.
* **Il calcolo dell'agente:** *"Se provo a scalare la prima collina ripida rischio di ribaltarmi e prendere -500. Se invece resto fermo allo spawn o vado avanti e indietro in pianura, non muoio mai!"*
* **Come risolverla:** Premia in modo chiaro la velocità orizzontale in avanti ($v_x > 0$) e dai un premio bonus per la massima distanza raggiunta, punendo il timeout o lo stallo in salita.

---
## 5. Percorso Guidato a Livelli: Progetta la tua `custom_reward`

Ti proponiamo di costruire la tua formula matematica per gradi:
* **🥈 Livello 1 (Progresso ed Eventi di Base):** Gestisci gli eventi finali (`is_success`, `is_crashed`, `is_timeout`) e assegna un piccolo premio proporzionale alla velocità in avanti `vx`.
* **🥇 Livello 2 (Controllo Assetto e Beccheggio):** Aggiungi una penalità quadratica o lineare sull'inclinazione `abs(angle)` e sulla velocità angolare `abs(ang_vel)` per evitare che l'auto si impenni.
* **💎 Livello 3 (Aderenza e Salto Controllato):** Premia quando entrambe le ruote toccano il terreno (`rear_wheel == 1.0` e `front_wheel == 1.0`) e incentiva a frenare in volo per correggere l'assetto.


In [ ]:
def custom_reward(obs, action, is_success, is_crashed, is_timeout):
    """
    Progetta qui la tua funzione matematica di Reward per Hill Climb Racing!

    SENSORI FISICI (obs - 8 valori):
      dist_x, quota_y   -> Distanza orizzontale (metri, spawn=6.67m) e quota verticale
      vx, vy            -> Velocità orizzontale (vx > 0 avanza) e verticale
      angle, ang_vel    -> Angolo di beccheggio telaio (rad, 0=orizzontale) e velocità angolare
      rear_wheel, front_wheel -> Contatto suolo ruota post e ant (1.0 = a terra, 0.0 = in volo)

    AZIONE SCELTA (action):
      0 = Folle / Motore spento | 1 = Acceleratore (Gas) | 2 = Freno / Retromarcia

    EVENTI DI FINE EPISODIO (True / False):
      is_success  -> Traguardo degli 800 metri raggiunto con successo!
      is_crashed  -> Schianto letale del pilota a terra!
      is_timeout  -> Tempo scaduto o veicolo bloccato in salita
    """
    dist_x, quota_y, vx, vy, angle, ang_vel, rear_wheel, front_wheel = obs
    reward = 0.0

    # =========================================================================
    # SCRIVI QUI LA TUA FORMULA DI REWARD SHAPING (Modifica e sperimenta!)
    # =========================================================================

    # 1. Incentiva l'avanzamento in avanti lungo la collina
    if vx > 0:
        reward += 0.1 * vx
    else:
        reward -= 0.1 * abs(vx)  # Penalizza la retromarcia o scivolamento

    # 2. Penalizza l'inclinazione eccessiva del telaio per evitare il ribaltamento
    reward -= 0.2 * abs(angle)

    # 3. Grande premio se si raggiunge il traguardo degli 800m!
    if is_success:
        reward += 100.0

    # 4. Forte penalità se il pilota si schianta a testa in giù!
    if is_crashed:
        reward -= 100.0

    # 5. Penalità se l'episodio finisce bloccati in salita per tempo scaduto
    if is_timeout:
        reward -= 20.0

    return float(reward)

print("✅ Funzione custom_reward salvata! Passa alla cella successiva per avviare l'addestramento.")


---
## 6. Addestramento e Telemetria di Guida

Durante l'addestramento vedrai la tabella di *rollout* con tre indicatori chiave:
* **`rollout/ep_rew_mean`**: La media della **tua custom reward** negli ultimi episodi.
* **`rollout/ep_len_mean`**: Quanti step dura in media una corsa (ricorda: `< 80` = schianto rapido, `300-700` = scalata controllata).
* **`rollout/success_rate`**: La percentuale di scalate riuscite oltre gli 800m (`0.0` = 0%, `1.0` = 100%).

Al termine dell'addestramento verranno mostrati la **Dashboard di Telemetria**, la **Diagnosi del Driving Coach** e i **video di prova** ad alta definizione!


In [ ]:
# Avvia l'addestramento PPO con la tua custom_reward
# Hint: impostiamo 300.000 step di base (i primi scavalcamenti compaiono attorno ai 150k step, mentre verso i 300k si consolida una guida pulita ed efficace).
# Se il tasso di successo finale è >= 80%, verrà generato automaticamente anche un report completo alla fine della run.
risultati = play2learn_hillclimb.run_lab(
    reward_fn=custom_reward,
    mission="campagna_standard",
    total_timesteps=300_000,
    nome_run="addestramento_base"
)


---
## 7. 🏆 Livelli Sfida Avanzati: Metti alla prova il tuo Pilota!

Sei riuscito a superare il livello base con almeno l'**80% di scalate riuscite**? Ora scegli uno degli **Scenari Sfida** modificando la tua `custom_reward` e impostando il parametro `mission` nella cella sottostante:

1. 🏜️ **`mission="deserto_dune"` (Rumore Originale)**: Dune sabbiose irregolari generate con l'algoritmo Perlin classico. Frequente rischio di decolli imprevisti e impennate all'atterraggio.
2. 🏔️ **`mission="montagna_estrema"` (Pendenze Ripide)**: Salite accentuate e creste affilate. Richiede una dosatura chirurgica del gas per non cappottare all'indietro.
3. 🧗 **`mission="colline_ripide"` (Massima Difficoltà)**: Scoscendimenti vertiginosi e rampe al limite della trazione fisica. Sfida estrema per piloti esperti!
4. 🛣️ **`mission="autostrada_veloce"` (Terreno Dolce)**: Tracciato scorrevole con ondulazioni dolci. Ideale per testare la massima velocità raggiungibile e l'aderenza delle ruote.


In [ ]:
# Scegli la missione tra:
# "deserto_dune", "montagna_estrema", "colline_ripide", "autostrada_veloce"

risultati_sfida = play2learn_hillclimb.run_lab(
    reward_fn=custom_reward,
    mission="deserto_dune",
    total_timesteps=300_000,
    nome_run="sfida_deserto"
)


---
## 8. 📊 Reportistica, Debug e Confronto tra Esperimenti

Quando sperimenti con la funzione di reward, hai a disposizione strumenti professionali per archiviare e analizzare i tuoi risultati:

* 🎯 **Archivio Esperimenti di Successo (`runs/successi/`)**: Ogni run che raggiunge un tasso di completamento $\ge 80\%$ viene archiviata automaticamente con i suoi video di prova e la Scheda Tecnica riassuntiva in PDF vettoriale.
* ⭐ **I Tuoi Preferiti (`runs/favorites/`)**: La tua galleria personale riservata ai modelli e alle corse che decidi tu di salvare manualmente (`risultati.salva_preferito(...)`).
* 🎞️ **Timelapse Evoluzione (`risultati.mostra_evoluzione()`)**: Animazione GIF che mostra la progressione del pilota nelle varie fasi dell'apprendimento sullo stesso percorso collinare.
* 📈 **Confronto tra run**: Visualizza a colpo d'occhio come le modifiche matematiche alla reward hanno aumentato la distanza media e la percentuale di successo.


In [ ]:
# 1. Visualizza il timelapse GIF dell'evoluzione dell'apprendimento nel tempo
risultati.mostra_evoluzione()


In [ ]:
# 2. Salva la run completata (o il suo video migliore) nei tuoi preferiti personali
risultati.salva_preferito(
    nome_personalizzato="Mia_Scalata_Top",
    note="Versione stabile: ottimo controllo di trazione e beccheggio sulle creste"
)


In [ ]:
# 3. Visualizza la galleria dei tuoi preferiti personali
play2learn_hillclimb.mostra_preferiti()


In [ ]:
# 4. Genera la Scheda Tecnica dell'esperimento (PDF, HTML e Markdown)
# I file vengono salvati nella cartella della run per poter essere consultati e stampati
report = risultati.genera_report(
    autore="Nome Studente / Gruppo",
    titolo="Esperimento Scalata Hill Climb Racing",
    note="Inserisci qui le tue osservazioni sul comportamento dell'agente e sulle modifiche alla reward"
)


In [ ]:
# 5. Metti a confronto due o più run per analizzare l'impatto delle modifiche matematiche
play2learn_hillclimb.confronta_run([risultati, risultati_sfida])


---
## 9. 🕒 Reportistica da Timestamp (Recupera Esperimenti Passati)

Hai eseguito più esperimenti nel corso della sessione o nei giorni scorsi e desideri rigenerare la **Scheda Tecnica** o rivedere i video di una run precedente?  
Non serve riaddestrare l'agente! Ti basta inserire il **timestamp** (o nome) della run che desideri analizzare.

### Come funziona:
1. **Consulta la cronologia**: Esegui la prima cella sottostante per visualizzare l'elenco di tutte le run salvate con i relativi timestamp (es. `20261117_010530`).
2. **Scegli il livello di dettaglio dal menu a tendina**:
   * **`Base`**: Genera istantaneamente la Scheda Tecnica ad alta definizione (PDF vettoriale, HTML, Markdown) e mostra i video già registrati per quella run.
   * **`Avanzata`**: Ricarica il modello addestrato ed esegue **nuove run di verifica**, generando e salvando **più video di test** con un'analisi telemetrica ancora più approfondita!


In [ ]:
# 1. Visualizza la cronologia di tutti gli esperimenti per trovare il timestamp della run
play2learn_hillclimb.mostra_cronologia_run()


In [ ]:
# @title 📊 Generatore di Report da Timestamp { run: "auto" }
# @markdown Inserisci il **timestamp** della run passata (es. `20261117_010530` oppure `latest`) e seleziona il livello di reportistica:
timestamp_run = "latest"  # @param {type:"string"}
tipo_report = "Base"  # @param ["Base", "Avanzata (genera e salva più video e run)"]
autore = "Studente"  # @param {type:"string"}
note = ""  # @param {type:"string"}

# Genera il report per la run selezionata (Base o Avanzata)
report = play2learn_hillclimb.genera_report_da_timestamp(
    timestamp=timestamp_run,
    tipo=tipo_report,
    autore=autore,
    note=note
)


### 📈 Confronto Diretto tra Run Passate (tramite Timestamp)

Vuoi mettere a confronto due esperimenti precedenti per analizzare come sono cambiati il tasso di completamento e la distanza percorsa?  
Inserisci i due timestamp nelle due celle seguenti: il sistema metterà a confronto i risultati estraendo telemetria e grafici comparativi!


In [ ]:
# @title 🕒 1. Inserisci il Timestamp del Primo Esperimento (Run A) { run: "auto" }
# @markdown Inserisci il timestamp del primo esperimento da confrontare (es. `20261117_010530` oppure `latest`):
timestamp_1 = "latest"  # @param {type:"string"}
print(f"✅ Primo esperimento impostato: {timestamp_1}")


In [ ]:
# @title 📊 2. Inserisci il Timestamp del Secondo Esperimento (Run B) e Avvia il Confronto { run: "auto" }
# @markdown Inserisci il timestamp del secondo esperimento (es. `20261117_004520` oppure `previous`):
timestamp_2 = "previous"  # @param {type:"string"}

# Unione automatica dei timestamp e avvio del confronto comparativo
timestamp_confronto = f"{timestamp_1.strip()}, {timestamp_2.strip()}"
play2learn_hillclimb.confronta_run(timestamp_confronto)


### 📦 Scarica l'Archivio degli Esperimenti (ZIP per Google Colab)

Vuoi scaricare sul tuo computer tutti i modelli addestrati, i grafici di telemetria, i report PDF e i video generati durante la sessione?  
Esegui la cella seguente per comprimere la cartella `runs/` in un file compresso `runs_hillclimb.zip` pronto per il download immediato:


In [ ]:
# Comprime la cartella 'runs/' e avvia automaticamente il download su Google Colab
play2learn_hillclimb.comprimi_runs(nome_zip="runs_hillclimb.zip")
